# Matchmaking Modeling and Policy Simulation

This notebook turns the EDA findings into a working ML/system prototype: outcome prediction, match-quality metrics, and offline policy comparison.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from matchmaking_system.data import load_games, prepare_match_records
from matchmaking_system.features import add_matchmaking_features, modeling_frame
from matchmaking_system.policies import Player
from matchmaking_system.simulation import simulate_policy

sns.set_theme(style='whitegrid')

In [ ]:
raw = load_games(ROOT / 'data/raw/games.csv')
matches = prepare_match_records(raw, include_draws=False)
matches = add_matchmaking_features(matches)
x, y = modeling_frame(matches)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=7, stratify=y)
x_train.shape, x_test.shape

## Baselines

A strong matchmaking project needs interpretable baselines. Elo expected score is the first baseline; logistic regression provides a calibrated ML baseline.

In [ ]:
def evaluate_model(name, y_true, probabilities):
    predictions = (probabilities >= 0.5).astype(int)
    return {
        'model': name,
        'roc_auc': roc_auc_score(y_true, probabilities),
        'accuracy': accuracy_score(y_true, predictions),
        'log_loss': log_loss(y_true, probabilities),
        'brier_score': brier_score_loss(y_true, probabilities),
    }

elo_probs = x_test['elo_white_win_prob'].clip(0.001, 0.999)
results = [evaluate_model('elo_expected_score', y_test, elo_probs)]
pd.DataFrame(results)

In [ ]:
logistic = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000)),
])
forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=7, n_jobs=-1)

models = {
    'logistic_regression': logistic,
    'random_forest': forest,
}

for name, model in models.items():
    model.fit(x_train, y_train)
    probs = model.predict_proba(x_test)[:, 1]
    results.append(evaluate_model(name, y_test, probs))

metrics = pd.DataFrame(results).sort_values('brier_score')
metrics

In [ ]:
best_model_name = metrics.iloc[0]['model']
best_model = models.get(best_model_name)
if best_model is None:
    matches['model_white_win_prob'] = matches['elo_white_win_prob']
else:
    matches['model_white_win_prob'] = best_model.predict_proba(x)[:, 1]

matches['model_uncertainty'] = 1 - (matches['model_white_win_prob'] - 0.5).abs() * 2
matches[['elo_white_win_prob', 'model_white_win_prob', 'model_uncertainty']].describe()

## Match Quality Metric

A production matchmaking system should not optimize only for prediction. We define a match quality score that rewards uncertain, balanced outcomes and penalizes large skill gaps.

In [ ]:
matches['skill_fairness_score'] = (1 - matches['abs_rating_gap'] / 800).clip(0, 1)
matches['match_quality_score'] = 0.7 * matches['model_uncertainty'] + 0.3 * matches['skill_fairness_score']
matches[['skill_fairness_score', 'model_uncertainty', 'match_quality_score']].describe()

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=matches.sample(min(3000, len(matches)), random_state=7), x='abs_rating_gap', y='match_quality_score', hue='winner', alpha=0.5)
plt.title('Match quality declines as rating gap grows')
plt.tight_layout()

## Offline Policy Simulation

We create a synthetic queue from observed players, then compare simple candidate selection policies. This demonstrates how the ML layer plugs into a production decision loop.

In [ ]:
white_players = matches[['white_id', 'white_rating']].rename(columns={'white_id': 'player_id', 'white_rating': 'rating'})
black_players = matches[['black_id', 'black_rating']].rename(columns={'black_id': 'player_id', 'black_rating': 'rating'})
players_df = pd.concat([white_players, black_players]).dropna().groupby('player_id', as_index=False)['rating'].mean()
players_df = players_df.sample(min(1000, len(players_df)), random_state=7)
players_df['wait_seconds'] = ((players_df['rating'].rank(method='first') % 180) + 10).astype(int)
players_df['latency_ms'] = ((players_df['rating'].rank(method='first') % 120) + 20).astype(int)
players_df['churn_risk'] = (players_df['wait_seconds'] / 240).clip(0, 0.8)

queue_players = [
    Player(row.player_id, row.rating, int(row.wait_seconds), int(row.latency_ms), float(row.churn_risk))
    for row in players_df.itertuples(index=False)
]

policy_results = pd.DataFrame([
    simulate_policy(queue_players, 'random', rounds=300).__dict__,
    simulate_policy(queue_players, 'closest_skill', rounds=300).__dict__,
    simulate_policy(queue_players, 'multi_objective', rounds=300).__dict__,
])
policy_results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(data=policy_results, x='policy_name', y='average_abs_rating_gap', ax=axes[0])
axes[0].set_title('Lower is better')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=policy_results, x='policy_name', y='average_fairness_score', ax=axes[1])
axes[1].set_title('Higher is better')
axes[1].tick_params(axis='x', rotation=20)

sns.barplot(data=policy_results, x='policy_name', y='average_total_score', ax=axes[2])
axes[2].set_title('Higher is better')
axes[2].tick_params(axis='x', rotation=20)
plt.tight_layout()

## End-to-End Solution Design

Recommended system for the repo:

1. Candidate generation: find eligible players by region, game mode, platform, party state, and wait time.
2. Feature service: compute rating gap, predicted win probability, queue age, latency compatibility, and churn-risk proxy.
3. Scoring service: combine fairness, uncertainty, wait-time relief, and latency into a multi-objective score.
4. Policy engine: select the highest-scoring feasible match with guardrails such as max rating gap and max latency.
5. Feedback loop: log match assignments, outcomes, abandons, rematches, and retention to retrain models.
6. Experimentation: compare policies offline first, then A/B test with fairness and retention guardrails.

This notebook is intentionally simple enough to run locally while still showing ML modeling, metrics, policy design, and production thinking.